# Общая CNN для датасета МРТ

Сначала выполните `data_prep/dataset.ipynb`: он разбирает и чистит датасет и сохраняет манифест —
готовый набор train/test, где каждый снимок встречается один раз. Этот ноутбук только
обучает и оценивает модель по манифесту.

Уровень аугментации для каждого источника задан в `SOURCES` в `mri/config.py` (сейчас у всех — R).
Классы балансируются при обучении (`balance_classes`, `mri/balance.py`). Гиперпараметры — `TRAIN` там же.
Цель: precision и recall ≥ 0.85 по каждому классу.

Код — в пакете `mri/`; здесь только настройки запуска и результаты.

In [ ]:
import os

import cv2
import numpy as np
import torch
from torch.utils.data import DataLoader

from mri.balance import balanced_sampler, sample_labels
from mri.config import CLASSES, IMG_SIZE, SOURCES, TRAIN
from mri.model import simple_cnn
from mri.train import build_datasets, evaluate, fit, new_run_dir, predict, save_json, set_seed

# Гиперпараметры задаются в TRAIN (mri/config.py). Для разового эксперимента
# можно переопределить здесь, не меняя config.py: HP = {**TRAIN, "epochs": 5}
HP = dict(TRAIN)
NUM_WORKERS = int(os.environ.get("MRI_WORKERS", "4"))  # если DataLoader падает на Windows — поставьте 0

set_seed(HP["seed"])
torch.set_num_threads(4)
cv2.setNumThreads(1)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = device == "cuda"
RUN_DIR = new_run_dir()
print("Устройство:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")
print("Гиперпараметры:", HP)
print("Аугментация:", SOURCES, "Результаты:", RUN_DIR, flush=True)

## Данные

Снимки и разбиение берутся из манифеста (`data_prep/dataset.ipynb`); исключённые при разборе
снимки не используются. Одна модель обучается на train всех источников; test — только для
итоговой оценки, без аугментации. Предобработка одинаковая: grayscale, обрезка полей, 224×224.

Группы снимков (тот же снимок или соседние срезы серии) в train и test не пересекаются,
поэтому метрики на test честные. Идентификаторов пациентов нет: группы — лучшее приближение.

In [ ]:
train_data, test_data, test_table, counts = build_datasets(HP["expand"])
# Балансировка: редкие классы берутся чаще (mri/balance.py); без неё — обычное перемешивание.
sampler = balanced_sampler(train_data, seed=HP["seed"]) if HP["balance_classes"] else None
train_dl = DataLoader(train_data, batch_size=HP["batch"], shuffle=sampler is None, sampler=sampler,
                      num_workers=NUM_WORKERS, pin_memory=device == "cuda", persistent_workers=NUM_WORKERS > 0)
test_dl = DataLoader(test_data, batch_size=256, shuffle=False, num_workers=0, pin_memory=device == "cuda")

config = {"policies": SOURCES, "data_counts": counts, "hyperparameters": HP,
          "workers": NUM_WORKERS, "device": device, "torch": str(torch.__version__),
          "cuda": torch.version.cuda, "classes": list(CLASSES), "image_size": IMG_SIZE,
          "evaluation_note": "Train/test share no near-duplicate groups (data_prep); "
                             "no patient IDs, distant slices of one patient may still be split."}
save_json(config, RUN_DIR / "config.json")
print("Снимков train:", sum(c["train_images"] for c in counts), "примеров за эпоху:", len(train_data),
      "снимков test:", len(test_data), flush=True)
labels = sample_labels(train_data)
print("Примеров за эпоху по классам без балансировки:", dict(zip(CLASSES, np.bincount(labels).tolist())))
if sampler is not None:
    drawn = labels[list(balanced_sampler(train_data, seed=HP["seed"]))]  # та же первая эпоха; train_dl не затронут
    print("С балансировкой (пример одной эпохи):     ", dict(zip(CLASSES, np.bincount(drawn).tolist())))

## Модель
5 блоков «свёртка → нормализация → ReLU → уменьшение в 2 раза» (224 → 7), затем среднее по каждому каналу и линейный слой на 4 класса (`mri/model.py`).

In [ ]:
model = simple_cnn(dropout=HP["dropout"]).to(device)
print(sum(p.numel() for p in model.parameters()), "параметров")

## Обучение
AdamW + OneCycle; после каждой эпохи `history.csv` и `model.pt` сохраняются в папку запуска.

In [ ]:
history = fit(model, train_dl, HP["epochs"], device, RUN_DIR, config,
              lr=HP["lr"], weight_decay=HP["weight_decay"])

## Precision и recall на тесте
**Precision** — из снимков, которые модель отнесла к классу, доля верных.  
**Recall** — из снимков этого класса, доля найденных моделью.

In [ ]:
pred = predict(model, test_dl, device)
summary, by_class, confusion = evaluate(test_table, pred, RUN_DIR)
print("Итог на test (all — весь test, далее — по источникам):")
print(summary.round(4).to_string(index=False))
print(by_class.round(4).to_string(index=False))
print("Сохранено:", RUN_DIR, flush=True)